# 02 Silver - Regex Skill Extraction

This notebook mirrors the local regex extraction layer.

Local equivalents:

- `src/skill_observatory/transformations/skill_extraction.py`
- `src/skill_observatory/transformations/build_historical_regex_skills.py`
- DuckDB tables: `historical_regex_skill_matches`, `historical_regex_skills`

Fabric role:

- read Bronze ads
- apply a compact technology taxonomy with aliases
- write a Silver one-row-per-ad-skill Delta table

For the full project, the local DuckDB regex extractor remains the richer implementation. This notebook is intentionally readable and smaller so the Fabric pipeline can show the transformation pattern.

In [ ]:
from pyspark.sql import functions as F
from functools import reduce

BRONZE_TABLE = "historical_job_ads_bronze"
SILVER_TABLE = "historical_regex_skills_silver"

# Small Fabric-friendly taxonomy for demonstration. The local project has
# the fuller taxonomy and QA loop.
TECH_SKILLS = {
    "python": [r"\\bpython\\b"],
    "sql": [r"\\bsql\\b", r"\\bt-sql\\b", r"\\bpl/sql\\b"],
    "azure": [r"\\bazure\\b", r"\\bmicrosoft azure\\b"],
    "aws": [r"\\baws\\b", r"\\bamazon web services\\b"],
    "docker": [r"\\bdocker\\b"],
    "kubernetes": [r"\\bkubernetes\\b", r"\\bk8s\\b"],
    "power_bi": [r"\\bpower bi\\b", r"\\bpowerbi\\b"],
    "dbt": [r"\\bdbt\\b", r"\\bdata build tool\\b"],
    "databricks": [r"\\bdatabricks\\b"],
    "snowflake": [r"\\bsnowflake\\b"],
}

ads = spark.table(BRONZE_TABLE).select(
    "id",
    "publication_month",
    F.lower(F.concat_ws(" ", F.col("headline"), F.col("description_text"))).alias("search_text"),
)

print(f"Loaded Bronze table: {BRONZE_TABLE}")
print(f"Rows available for regex extraction: {ads.count():,}")

In [ ]:
skill_frames = []

for skill, patterns in TECH_SKILLS.items():
    combined_pattern = "|".join(f"({pattern})" for pattern in patterns)
    skill_frame = (
        ads
        .where(F.col("search_text").rlike(combined_pattern))
        .select(
            "id",
            "publication_month",
            F.lit(skill).alias("skill"),
            F.lit("regex").alias("skill_source"),
        )
    )
    skill_frames.append(skill_frame)

silver_df = reduce(lambda left, right: left.unionByName(right), skill_frames)
silver_df = silver_df.dropDuplicates(["id", "publication_month", "skill"])

silver_df.write.mode("overwrite").format("delta").saveAsTable(SILVER_TABLE)

print(f"Wrote Silver table: {SILVER_TABLE}")
print(f"Skill rows: {silver_df.count():,}")

In [ ]:
display(
    spark.table(SILVER_TABLE)
    .groupBy("skill")
    .count()
    .orderBy(F.desc("count"))
)